In [1]:
!pip install transformers accelerate datasets evaluate -q

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
from datasets import Dataset, DatasetDict
from evaluate import load
import torch
import warnings
warnings.filterwarnings('ignore')

print('GPU:', torch.cuda.is_available())
print('GPU Name:', torch.cuda.get_device_name(0))

df = pd.read_csv('../data/cleaned_data.csv')
df = df.dropna(subset=['text_clean'])
df['text_clean'] = df['text_clean'].astype(str)

labels = df['label'].unique().tolist()
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}
num_labels = len(label2id)
df['label_id'] = df['label'].map(label2id)

print('Data loaded:', df.shape)
print('Labels:', label2id)

GPU: True
GPU Name: NVIDIA GeForce RTX 4060 Laptop GPU
Data loaded: (51055, 4)
Labels: {'Anxiety': 0, 'Normal': 1, 'Depression': 2, 'Suicidal': 3, 'Stress': 4, 'Bipolar': 5, 'Personality disorder': 6}


In [4]:
f, test_df = train_test_split(test_df, test_size=0.50, random_state=42, stratify=test_df['label'])

train_df, test_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(test_df, test_size=0.50, random_state=42, stratify=test_df['label'])

def make_dataset(data):
    return Dataset.from_pandas(
        data[['text_clean', 'label_id']].rename(
            columns={'text_clean':'text', 'label_id':'label'}),
        preserve_index=False)

dataset = DatasetDict({
    'train': make_dataset(train_df),
    'validation': make_dataset(val_df),
    'test': make_dataset(test_df)
})

MODEL_NAME = 'j-hartmann/emotion-english-distilroberta-base'
print('Loading emotion-specific RoBERTa tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(examples):
    return tokenizer(examples['text'], truncation=True, max_length=128)

tokenized_dataset = dataset.map(tokenize, batched=True)
print('Tokenization done!')
print(tokenized_dataset)

Loading emotion-specific RoBERTa tokenizer...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Map:   0%|          | 0/35738 [00:00<?, ? examples/s]

Map:   0%|          | 0/7658 [00:00<?, ? examples/s]

Map:   0%|          | 0/7659 [00:00<?, ? examples/s]

Tokenization done!
DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 35738
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 7658
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 7659
    })
})


In [6]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=num_labels,
    id2label=id2label, 
    label2id=label2id,
    ignore_mismatched_sizes=True
)

training_args = TrainingArguments(
    output_dir='../models/saved/mental_roberta',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    logging_steps=50,
    learning_rate=1e-5,
    weight_decay=0.01,
    fp16=True,
)

metric = load('accuracy')
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

print('Starting Mental RoBERTa training on RTX 4060!')
trainer.train()
print('Training complete!')

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting Mental RoBERTa training on RTX 4060!


Epoch,Training Loss,Validation Loss,Accuracy
1,0.521590,0.551612,0.779969
2,0.432921,0.534541,0.788326
3,0.449180,0.506832,0.800601
4,0.336288,0.509465,0.802690
5,0.359886,0.513869,0.803735


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Training complete!


In [ ]:
trainer.save_model('../models/saved/mental_roberta')
tokenizer.save_pretrained('../models/saved/mental_roberta')
print('Mental RoBERTa saved!')
print(f'Best accuracy: 80.37%')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Mental RoBERTa saved!
Best accuracy: 80.37%


: 